# Chapter 17 &mdash; Boolean Functions and Why Truth Tables Do Not Scale

**Concept 1 of the Chapter 17 decomposition:** *Boolean Functions, Truth-Table Personalities, and Why Tables Do Not Scale*

There are $2^{2^N}$ functions of $N$ inputs; a 64-input table would need more rows than atoms in a city.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter17/Concept-Truth-Tables-Do-Not-Scale/Concept-Truth-Tables-Do-Not-Scale.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


An $N$-input Boolean function is fixed by its **truth table**: $2^N$ rows. So there are
$2^{2^N}$ distinct functions &mdash; 16 for $N=2$, 256 for $N=3$, and for $N=6$ already
more than $10^{19}$.

The table itself is the problem. At $N=64$ a truth table has $1.8\times10^{19}$ rows;
at one byte per row that is 18 exabytes.

Yet the functions we actually care about &mdash; adders, comparators, control logic &mdash;
have enormous **regularity**. The question the chapter answers is how to exploit it:
represent the function by a structure whose size tracks its **complexity** rather than
its **arity**.

The answer, surprisingly, is a **minimal DFA**.

## 2. Definitions

### Counting

In [ ]:
def n_functions(N): return 2 ** (2 ** N)
def table_rows(N):  return 2 ** N

def truth_table(f, N):
    from itertools import product
    return [(bits, f(bits)) for bits in product([0, 1], repeat=N)]

### Some regular functions, for contrast

In [ ]:
def AND_n(bits):  return int(all(bits))
def PARITY(bits): return sum(bits) % 2
def MAJORITY(bits): return int(sum(bits) * 2 > len(bits))

<!-- nav-strip -->

---

&larr;&nbsp;[Ch16&nbsp;18.&nbsp;Variable Ordering, and What BDDs Do Not Settle](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-Variable-Ordering-And-P-Versus-NP/Concept-Variable-Ordering-And-P-Versus-NP.ipynb) &nbsp;&middot;&nbsp; [**Chapter 17** index](https://github.com/ganeshutah/Jove/blob/master/Chapter17/README.md) &nbsp;&middot;&nbsp; [Ch17&nbsp;2.&nbsp;A BDD is the Minimal DFA of a Function's On-Set](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter17/Concept-BDD-As-Minimal-DFA/Concept-BDD-As-Minimal-DFA.ipynb)&nbsp;&rarr;

---

## 3. Tests

The double exponential.

In [ ]:
print("%-4s %-14s %s" % ("N", "table rows", "distinct functions"))
for N in [1, 2, 3, 4, 6]:
    print("%-4d %-14s %s" % (N, format(table_rows(N), ','), format(n_functions(N), ',')))
assert n_functions(6) > 10 ** 19

At $N=64$ the table is unusable.

In [ ]:
for N in [16, 32, 64]:
    rows = table_rows(N)
    print("  N=%2d : %s rows, %.3g bytes at one byte per row"
          % (N, format(rows, ','), float(rows)))
assert table_rows(64) > 1.8e19

But the functions we build hardware from are **regular**.

In [ ]:
for name, f in [('AND', AND_n), ('PARITY', PARITY), ('MAJORITY', MAJORITY)]:
    t = truth_table(f, 3)
    print("  %-9s :" % name, ''.join(str(v) for _, v in t))
print("\nEach is describable in a sentence -- the table is a bad encoding of that.")

Regularity shows up as **repeated subfunctions**.

In [ ]:
from itertools import product
def subfunctions(f, N):
    # fix the first variable; how many distinct residual functions are there?
    subs = set()
    for first in [0, 1]:
        subs.add(tuple(f((first,) + rest) for rest in product([0, 1], repeat=N - 1)))
    return subs
for name, f in [('AND', AND_n), ('PARITY', PARITY), ('MAJORITY', MAJORITY)]:
    print("  %-9s distinct residual functions after fixing x1 : %d"
          % (name, len(subfunctions(f, 4))))
print("\nTwo, always -- and that is what a BDD exploits.")

The plan for the chapter.

In [ ]:
print("truth table : size 2^N always")
print("BDD         : size tracks the number of DISTINCT residual functions")
print()
print("And 'distinct residual function' is exactly Myhill-Nerode's")
print("'distinguishable state'.  Concept 2 makes that identification.")

## 4. Exercises


1. How many 4-input functions are there? Write the number out.
2. Which 2-input functions are *not* expressible with AND, OR, NOT? (Trick question.)
3. Give a function of $N$ inputs with $2^{N-1}$ distinct residuals.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter17/Concept-Truth-Tables-Do-Not-Scale')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')